In [ ]:
#캠페인 database에 불러오기 성공
import time
import random
import requests
import psycopg2
import signaturehelper

# 네이버 API 요청에 필요한 HTTP 헤더를 생성
def get_header(method, uri, api_key, secret_key, customer_id):
    timestamp = str(round(time.time() * 1000))
    signature = signaturehelper.Signature.generate(timestamp, method, uri, secret_key)
    return {'Content-Type': 'application/json; charset=UTF-8', 'X-Timestamp': timestamp, 'X-API-KEY': API_KEY, 'X-Customer': str(CUSTOMER_ID), 'X-Signature': signature}

​
BASE_URL = 'https://api.searchad.naver.com'
API_KEY = '0100000000035ba1cc76db8c4cba24a05c3386d0f6d65e4245c883e3e70d24d46fd57193bb'
SECRET_KEY = 'AQAAAAADW6HMdtuMTLokoFwzhtD2hVgWxTOrgKQdi8s+So+KlA=='
CUSTOMER_ID = '1673134'
​
def naver_campaign_data():
    # 네이버 검색광고 API 호출 URL 및 헤더 설정
    uri = '/ncc/campaigns'
    method = 'GET'
    headers = get_header(method, uri, API_KEY, SECRET_KEY, CUSTOMER_ID)
    response = requests.get(BASE_URL + uri, headers=headers)
    
    if response.status_code == 200:
        data = response.json()  # API로부터 받은 데이터를 JSON 형태로 파싱
        return data
    else:
        print("API 호출에 실패했습니다.")
        print("상태 코드:", response.status_code)
        return None
​
def insert_data_into_sql(db_info, data):
    # PostgreSQL 데이터베이스에 연결
    connection = None
    cursor = None
    try:
        connection = psycopg2.connect(
            host=db_info['host'],
            user=db_info['user'],
            password=db_info['password'],
            dbname=db_info['database']
        )
        cursor = connection.cursor()
        
        # 데이터 삽입 또는 업데이트 쿼리 작성
        insert_query = """
            INSERT INTO NaverAPI_campaign (nccCampaignId, customerId, name, userLock, campaignTp, deliveryMethod, trackingUrl, trackingMode, usePeriod, dailyBudget, useDailyBudget, totalChargeCost, status, statusReason, expectCost, migType, delFlag, regTm, editTm) 
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (nccCampaignId) DO UPDATE SET
                customerId = EXCLUDED.customerId,
                name = EXCLUDED.name,
                userLock = EXCLUDED.userLock,
                campaignTp = EXCLUDED.campaignTp,
                deliveryMethod = EXCLUDED.deliveryMethod,
                trackingUrl = EXCLUDED.trackingUrl,
                trackingMode = EXCLUDED.trackingMode,
                usePeriod = EXCLUDED.usePeriod,
                dailyBudget = EXCLUDED.dailyBudget,
                useDailyBudget = EXCLUDED.useDailyBudget,
                totalChargeCost = EXCLUDED.totalChargeCost,
                status = EXCLUDED.status,
                statusReason = EXCLUDED.statusReason,
                expectCost = EXCLUDED.expectCost,
                migType = EXCLUDED.migType,
                delFlag = EXCLUDED.delFlag,
                regTm = EXCLUDED.regTm,
                editTm = EXCLUDED.editTm
        """
        # 위에 데이터 sql 넣을 때 어떻게 넣을거야? Update? truncate? insert 등 고민 필요


        # 데이터 삽입 반복
        for record in data:
            # 날짜 데이터 타입 변환
            reg_tm = record.get('regTm')
            edit_tm = record.get('editTm')
            
            # 변수가 none이 아니라면 조건문에서 true 쳐줌.
            if reg_tm:
                reg_tm = reg_tm.replace('T', ' ').replace('Z', '')
            if edit_tm:
                edit_tm = edit_tm.replace('T', ' ').replace('Z', '')
            
            ### reg_tm / edit_tm 이 api로 어떤 형식으로 가져오고 있어? 확인
            ### 그거 t 랑 z 꼭 없어야해?? 형식 문제있어??? 없으면 그냥 가져와도 돼

            ### api로 가져온 데이터 db에 넣을때 데이터 넣은 시간, 데이터 업데이트 시간 기록 필요 (히스토리 차원에서 대체적으로 있는게 좋음)


            # trackingUrl 필드가 없을 경우 빈 문자열로 설정
            tracking_url = record.get('trackingUrl', '')
            
            # 데이터 삽입 또는 업데이트 실행
            try:
                cursor.execute(insert_query, (
                    record['nccCampaignId'],
                    record['customerId'],
                    record['name'],
                    record['userLock'],
                    record['campaignTp'],
                    record['deliveryMethod'],
                    tracking_url,
                    record['trackingMode'],
                    record['usePeriod'],
                    record['dailyBudget'],
                    record['useDailyBudget'],
                    record['totalChargeCost'],
                    record['status'],
                    record['statusReason'],
                    record['expectCost'],
                    record['migType'],
                    record['delFlag'],
                    reg_tm,
                    edit_tm
                ))
            except Exception as e:
                print(f"데이터 삽입 오류 발생: {e}")
        
        # 변경 사항을 실제 데이터베이스에 반영
        connection.commit()
        print("데이터가 커밋되었습니다.")
    except (Exception, psycopg2.DatabaseError) as error:
        print("데이터베이스 오류 발생:", error)
    finally:
        # 커서 종료 및 연결 종료
        if cursor is not None:
            cursor.close()
        if connection is not None:
            connection.close()
​
# SQL 데이터베이스 연결 정보 입력
db_info = {
    'host': 'db1-dev.postgres.database.azure.com',
    'user': 'dev',
    'password': '@Vkdlxhdnpdl0',
    'database': 'testDB_241128'
}
​
# 네이버 API 데이터 가져오기
naver_ads_data = naver_campaign_data()
​
if naver_ads_data:
    print("가져온 데이터:", naver_ads_data)  # 데이터 출력하여 확인
    # SQL 테이블에 데이터 삽입
    insert_data_into_sql(db_info, naver_ads_data)
    print("데이터 삽입 완료")
else:
    print("데이터가 없어서 삽입하지 못했습니다.")

In [ ]:
# 검색어 데이터 불러오기 성공
 
import time
import requests
# import psycopg2
from datetime import datetime
import signaturehelper


BASE_URL = 'https://api.searchad.naver.com'
API_KEY = '0100000000035ba1cc76db8c4cba24a05c3386d0f6d65e4245c883e3e70d24d46fd57193bb'
SECRET_KEY = 'AQAAAAADW6HMdtuMTLokoFwzhtD2hVgWxTOrgKQdi8s+So+KlA=='
CUSTOMER_ID = '1673134'


def get_header(method, uri, api_key, secret_key, customer_id):
    timestamp = str(round(time.time() * 1000))
    signature = signaturehelper.Signature.generate(timestamp, method, uri, SECRET_KEY)
    return {'Content-Type': 'application/json; charset=UTF-8', 'X-Timestamp': timestamp, 'X-API-KEY': API_KEY, 'X-Customer': str(CUSTOMER_ID), 'X-Signature': signature}

# 데이터베이스 연결 정보
db_info = {
    'host': 'db1-dev.postgres.database.azure.com',
    'user': 'dev',
    'password': '@Vkdlxhdnpdl0',
    'database': 'testDB_241128'
}

def get_shopping_conversion_data():
    try:
        # 네이버 검색광고 대용량 다운로드 보고서 생성
        uri = '/stat-reports'
        method = 'POST'
        headers = get_header(method, uri, API_KEY, SECRET_KEY, CUSTOMER_ID)
        paras = {
            'reportTp': 'SHOPPINGKEYWORD_CONVERSION_DETAIL',
            'statDt': '20241125'
        }
        print("보고서 생성 요청 중...")
        response = requests.post(BASE_URL + uri, headers=headers, json=paras)
        
        # {'key' : 'value'} 똑같은 형식인데 웹 통신에선 json이라고 부르고 파이썬에서는 딕셔너리라고 부름.
        ## json() 이거가 딕셔너리 만들어주기? json 형식의 데이터를 dict로 만들어줌.
        if response.status_code == 200:
            report_job_id = response.json().get('reportJobId')

            ## if 다음엔 true 만 실행. not 은 부정. false 일때 실행한다. 
            ## report_job_id 에 값이 있으면 true 없으면 false
            if not report_job_id:
                print("reportJobId를 가져오지 못했습니다.")
                return None
            print(f"보고서 생성 성공. Report Job ID: {report_job_id}")
        else:
            print("보고서 생성 API 호출에 실패했습니다.")
            print("상태 코드:", response.status_code)
            return None

        # 보고서가 준비될 때까지 대기
        print("보고서 준비 중입니다...")
        for attempt in range(30):  # 최대 30번까지 시도 (대기 시간 포함)
            uri = f'/stat-reports/{report_job_id}'
            headers = get_header('GET', uri, API_KEY, SECRET_KEY, CUSTOMER_ID)
            response = requests.get(BASE_URL + uri, headers=headers)
            
            ## 만약 아까 위에서 보고서 생성했다면 (=status code가 200이라면)
            if response.status_code == 200:
                print(response.json())
                ## status = 보고서 상태값. 생성완료인지 생성중인지 알 수 있음. (보고서 생성되었으면 무조건 생성완료라고 뜨는걸까?)
                ## 보고서 생성 요청 했은데 아직 검색광고 창에서 반영이 안되었거나, 다운로드 버튼이 없을수도 있기 때문
                status = response.json().get('status')
                print(f"현재 보고서 상태: {status} (시도 횟수: {attempt + 1})")
                ## status 가 built = 생성완료
                ## 생성완료 되었으면 진행시켜!
                ## 보고서 읽어서 download url 가져오기
                if status == 'BUILT':
                    download_url = response.json().get('downloadUrl')
                    if download_url:
                        print("보고서 준비 완료. 다운로드 URL 확보.")
                        break # 보고서 다운 했으니까 이제 attempt 반복 안해도댐 break / continue 는 for 문 안에서 쓴다. continue 는 다음줄 실행 안하고 다시 처음으로 
                # 만약 아직 생성완료가 안되었다면, 10초 기다렸다가 다운로드 버튼 생기면 다시 재시도 한다. 
                elif status == 'REGIST':
                    time.sleep(10)  # 보고서 준비 중, 10초 대기 후 재시도
                else:
                    print(f"보고서 상태 오류: {status}")
                    return None
            else:
                print("보고서 다운로드 API 호출에 실패했습니다.")
                print("상태 코드:", response.status_code)
                return None
        # else: ## 얘 이상해.... 없어야 할 거 같음. 
        #     print("보고서 준비가 완료되지 않았습니다.")
        #     return None

        # 다운로드 URL을 사용하여 데이터 가져오기
        print("보고서 다운로드 중...")
        uri = '/report-download/'
        method = 'GET'
        headers = get_header(method, uri, API_KEY, SECRET_KEY, CUSTOMER_ID)
        params = {
            'authtoken': download_url.split('?')[-1].split('=')[-1]
        }
        response = requests.get(BASE_URL + uri, headers=headers, params=params)
        
        if response.status_code == 200:
            print("보고서 다운로드 성공.")
            print()
            print(response.text.splitlines()) ##문자열을 줄바꿈 해주기? \n이라고 원래 되어있었나..? 문자열을 \n으로 구분해서 각각 문자열로 만든 뒤 모두 리스트에 넣음.
            return response.text.splitlines()
        else:
            print("보고서 다운로드 실패.")
            print("상태 코드:", response.status_code)
            return None
    except Exception as e:
        print(f"get_shopping_conversion_data 함수에서 오류 발생: {e}")
        return None

# 함수 호출
# 이거 왜 필요해? 없어도 되려나? 함수 정의 안에서 유효성 검증을 이미 했으므로 삭제해도 될 듯.
# if data:
#     print("다운로드된 데이터:")
#     for line in data[:5]:  # 데이터가 많을 경우 일부만 출력 (없어도 될거같음)
#         print(line)
# else:
#     print("데이터를 가져오지 못했습니다.")


def insert_data_into_sql(db_info, table_name, data, insert_query):
    connection = None
    cursor = None
    try:
        # 데이터베이스 연결
        connection = psycopg2.connect(
            host=db_info['host'],
            user=db_info['user'],
            password=db_info['password'],
            dbname=db_info['database']
        )
        cursor = connection.cursor()
        print(f"{table_name} 테이블에 데이터 삽입 시작...")
        
        # 데이터를 반복하면서 삽입
        ## enumerate 를 사용하면 for문에서 2개를 사용할 수 있다. 
        ## for문의 첫번째는 리스트의 인덱스! (리스트 순서, 즉 0,1,2,3,,,)
        ## for문의 두번째는 리스트의 값! (리스트안의 첫번째 값(즉 list[0]에 해당하는 값), 두번째 값,,,)
        ## 그래서 enumerate가 원하는게 뭔데? record 값??? = index 랑 원소값 둘다 for문에서 사용하고 싶어서.  리스트의 원소에는 앞에서 차례대로, (그리고 자동으로) 인덱스가 매겨진다.
        ## 인덱스는 0부터 시작한다. 즉 원소는, 원소의 값 뿐만 아니라 인덱스도 가진다. 리스트에 [인덱스] 를 하면 인덱스에 해당하는 원소를 가져온다.
        ## 그런데 아래의 반복문에서 인덱스는 활용되지 않았다. 따라서 enumerate는 안써도 된다.
        for index, record in enumerate(data): 
            try:
                # 현재 시간 추가 (reg_tm, edit_tm)
                reg_tm = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                edit_tm = reg_tm
                record.extend([reg_tm, edit_tm])
                cursor.execute(insert_query, record)
                print(f"{table_name} 테이블에 데이터 삽입 성공: {record}")
            except psycopg2.IntegrityError as e:
                connection.rollback()  # 중복 키 오류 발생 시 롤백 처리 후 넘어감
                print(f"{table_name} 테이블에 중복된 데이터가 있습니다: {e}")
            except Exception as e:
                connection.rollback()  # 다른 오류 발생 시 롤백 처리 후 넘어감
                print(f"{table_name} 테이블에 데이터 삽입 오류 발생: {e}")
        
        # 모든 데이터 삽입 후 변경 사항 커밋
        connection.commit()
        print(f"{table_name} 테이블에 데이터 삽입 성공")
    except (Exception, psycopg2.DatabaseError) as error:
        print(f"데이터베이스 오류 발생: {error}")
    finally:
        # 커서 및 연결 종료
        if cursor is not None:
            cursor.close()
        if connection is not None:
            connection.close()
        print(f"{table_name} 테이블에 대한 데이터베이스 연결 종료")

# 쇼핑 검색어 데이터 삽입 함수
def insert_shopping_conversion_table(data):
    ## if not = 만약 false 라면 진행시켜!
    ## data = get shopping conversion data (api호출로 받은 값)
    ## data 안에 값이 있다면 true, 없다면 false
    ## 즉, api로 불러와진 검색어 데이터가 없다면 진행시켜!
    if not data: 
        print("삽입할 데이터가 없습니다.")
        return
    
    print("쇼핑 검색어 데이터 삽입 시작...")
    # 테이블에 삽입할 데이터 파싱 및 SQL 정의
    parsed_data = [] # 빈 리스트 변수 정의
    for idx, line in enumerate(data):  # 첫 번째 줄 포함하여 모든 데이터 사용
        columns = line.split('\t')  # \t로 구분되어 있던 string을 탭으로 구분
        if len(columns) != 15: #리스트 안의 값은 전부 동일해야함. 
            print(f"데이터 형식 오류 (줄 번호 {idx + 1}): 예상 필드 개수는 15개인데, {len(columns)}개를 받았습니다. 데이터: {line}")
            continue # 만약 15개 아니라면 버릴거야. 취급도 안할거야
        parsed_data.append(columns) ## 리스트를 빈 리스트에 넣어주기. 리스트 안의 리스트. 이건 테이블 형식으로 처리해주려고 사용. 다만 리스트 안의 모든 리스트는 원소 개수가 동일해야함. 

    # 파싱된 데이터의 개수 출력
    print(f"파싱된 데이터 개수: {len(parsed_data)}")
    if len(parsed_data) == 0:
        print("파싱된 데이터가 없습니다. 데이터 형식에 문제가 있을 수 있습니다.")
        return

    insert_query = """
        INSERT INTO naverapi_shoppingkeyword_conversion_detail (
            date, customerId, nccCampaignId, adGroupId, searchKeyword,
            adId, businessId, hour, regionCode, mediaCode, pcMobileType,
            conversionMethod, conversionType, conversionCount, salesConversion,
            reg_tm, edit_tm
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    insert_data_into_sql(db_info, 'naverapi_shoppingkeyword_conversion_detail', parsed_data, insert_query)
    print("쇼핑 검색어 데이터 삽입 완료.")

# 함수 호출
## 변수에 값을 저장해주려고
data = get_shopping_conversion_data()
## 어짜피 값 볼거 아니고, db로 넣는거 성공 or 페일로 끝나니까 변수 저장 안해도 됌. 실행만 할거임. 
insert_shopping_conversion_table(data)


# 함수 안에서 정의된 변수는 함수 밖에서 사용하려고 하면 오류 날것.
